# Sparsity: every run evaluated with 32 and 16 laser beams (one-off)

Runtime -> Change runtime type -> **GPU** (T4 is enough). Set cell 1, then **Runtime -> Run all**.

For every finished run on Drive (NDT-Transformer `ndt`, BEVPlace++ `bev`, own NDT network `myndt`, and the official
BEVPlace++ checkpoint) this runs the evaluation type `sparsity` (`src/lpr/evals.py`) on its last checkpoint, without
retraining:

- **64 beams**: the run's own variant (same as `seqs_heading`);
- **`both`**: map and queries from the 32 / 16-beam variant (a cheaper LiDAR everywhere);
- **`query`**: map from 64 beams, queries from 32 / 16 beams (a cheaper LiDAR localising in a good map).

The sparse variants (`<date>_beams32_<hash>`, `<date>_beams16_<hash>`) were built on Azure; this notebook copies them
from the blob container to Drive and to this machine (SAS URL in Colab Secrets as `AZURE_SAS`, read + list is enough).
Results: `<run>/eval/sparsity/<checkpoint>/` (`summary.csv`, `sparsity.png`), and the table at the end
(`MyDrive/lpr/sparsity_all_runs.csv`). A complete evaluation is not repeated, so Run all again after a disconnect.

## 1. Settings

In [ ]:
REPO_URL = "https://github.com/dov85/lidar-place-recognition"
DRIVE = "/content/drive/MyDrive/lpr"
KITTI = "/content/kitti"                 # poses only
METHODS = ["ndt", "myndt", "bev"]        # which kinds of runs
STATUS = ["finished"]                    # runs with this status in their config.json
INCLUDE_RUNS = []                        # extra run folders, whatever their status, e.g. ["2026-10-19_de5611"]
EXCLUDE_RUNS = []                        # run folders to skip
EVAL_CKPT = "last"                       # "last" or a checkpoint file name, the same for every run
OFFICIAL_BEV = True                      # also the authors' BEVPlace++ checkpoint (runs/bev/official_<variant>)
PARAMS = {}                              # overrides of the sparsity parameters, e.g. {"beams": [16], "modes": ["both"]}

## 2. Drive, GPU, code, poses

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')
import os, sys, json, glob, subprocess, importlib
!nvidia-smi --query-gpu=name,memory.total --format=csv
url = REPO_URL
try:  # private repo: store a token in Colab Secrets as GITHUB_TOKEN
    url = REPO_URL.replace("https://", f"https://{userdata.get('GITHUB_TOKEN')}@")
except Exception:
    pass
if not os.path.exists("/content/lpr"):
    subprocess.run(["git", "clone", "-q", url, "/content/lpr"], check=True)
else:
    subprocess.run(["git", "-C", "/content/lpr", "pull", "-q"], check=True)
%cd /content/lpr
!git log --oneline -1
!bash scripts/setup_third_party.sh
!pip install -q -r requirements.txt
sys.path.insert(0, "/content/lpr/src")
import lpr.experiments as ex, lpr.evals as ev
importlib.reload(ex); importlib.reload(ev)
missing_poses = [s for s in ev.ALL_SEQS if not os.path.exists(f"{KITTI}/poses/{s}.txt")]
if missing_poses:
    !python scripts/download_kitti.py --out {KITTI} --seqs {" ".join(missing_poses)} --poses_only
SP = ev.full_eval_params("sparsity", PARAMS)
print("sparsity parameters:", SP)

## 3. The runs

In [ ]:
import pandas as pd
OFFICIAL_CKPT = "/content/lpr/third_party/BEVPlace2/runs/Aug08_10-17-29/model_best.pth.tar"
DATA = {"ndt": "ndt", "myndt": "ndt", "bev": "bev"}     # whose variants a method uses
runs = []
for m in METHODS:
    for cfg_path in sorted(glob.glob(f"{DRIVE}/runs/{m}/*/config.json")):
        run_dir = os.path.dirname(cfg_path)
        name = os.path.basename(run_dir)
        cfg = json.load(open(cfg_path))
        official = m == "bev" and name.startswith("official_")
        if name in EXCLUDE_RUNS or (official and not OFFICIAL_BEV):
            continue
        if cfg.get("status") not in STATUS and name not in INCLUDE_RUNS and not official:
            continue
        if official:
            ckpt = OFFICIAL_CKPT
        elif glob.glob(f"{run_dir}/epoch_*.pth"):
            ckpt = EVAL_CKPT
        else:
            continue
        variant = cfg.get(f"{DATA[m]}_variant")
        if not variant:
            print("no variant in", cfg_path, "(skipped)")
            continue
        runs.append(dict(method=m, run=name, run_dir=run_dir, ckpt=ckpt, variant=variant,
                         data_dir=f"/content/{DATA[m]}/{variant}", status=cfg.get("status")))
RUNS = pd.DataFrame(runs)
print(len(RUNS), "runs")
display(RUNS[["method", "run", "status", "variant"]])

## 4. Data: the runs' 64-beam variants and their 32 / 16-beam variants
Every sequence of every variant is copied to this machine (`/content/ndt/<variant>/`, `/content/bev/<variant>/`);
what is not on Drive yet is copied from the blob container first.

In [ ]:
SAS = userdata.get("AZURE_SAS")
for (dm, variant), _ in RUNS.groupby([RUNS.method.map(DATA), "variant"]):
    vdir, local = f"{DRIVE}/{dm}/{variant}", f"/content/{dm}/{variant}"
    tar = lambda s: f"{vdir}/{ex.archive_name(dm, s)}"
    missing = [s for s in SP["seqs"] if not os.path.exists(tar(s))]
    if missing:
        res = subprocess.run(["python", "scripts/fetch_from_azure.py", "--sas_url", SAS, "--method", dm,
                              "--variant", variant, "--seqs", *missing, "--drive", DRIVE],
                             capture_output=True, text=True, cwd="/content/lpr")
        print(res.stdout, res.stderr[-2000:])
    ext = "000000.png" if dm == "bev" else "000000.bin"
    for s in SP["seqs"]:
        if os.path.exists(tar(s)) and not os.path.exists(f"{local}/{s}/{ext}"):
            os.makedirs(local, exist_ok=True)
            subprocess.run(["tar", "-xf", tar(s), "-C", local], check=True)
    if not os.path.exists(f"{local}/config.json") and os.path.exists(f"{vdir}/config.json"):
        subprocess.run(["cp", f"{vdir}/config.json", f"{local}/config.json"], check=True)
    print(f"{dm} {variant}: on this machine for", [s for s in SP["seqs"] if os.path.exists(f"{local}/{s}/{ext}")])
    res = subprocess.run(["python", "scripts/fetch_sparse.py", "--method", dm, "--variant", variant,
                          "--drive", DRIVE, "--data_root", f"/content/{dm}", "--sas_url", SAS,
                          "--beams", *map(str, SP["beams"]), "--seqs", *SP["seqs"]],
                         capture_output=True, text=True, cwd="/content/lpr")
    print(res.stdout, res.stderr[-2000:])

## 5. Evaluations
One `sparsity` evaluation per run (64 beams, then every number of beams in both modes). Already complete ones are
skipped; a disconnected one continues with the sequences that are missing.

In [ ]:
import time
for r in RUNS.itertuples():
    t0 = time.time()
    res = subprocess.run(["python", "scripts/evaluate.py", "--method", r.method, "--run_dir", r.run_dir,
                          "--ckpt", r.ckpt, "--eval_type", "sparsity", "--data_dir", r.data_dir,
                          "--kitti_root", KITTI, "--params", json.dumps(PARAMS)],
                         capture_output=True, text=True, cwd="/content/lpr")
    out = res.stdout if res.returncode == 0 else res.stdout[-2000:] + res.stderr[-3000:]
    print(f"===== {r.method} {r.run} ({(time.time() - t0) / 60:.1f} min)")
    print("\n".join(out.splitlines()[-14:]))

## 6. Tables and plots: Recall@1 (%) per run, number of beams and mode

In [ ]:
from IPython.display import Image, display
!python scripts/summarize_runs.py --drive {DRIVE} > /dev/null
e = pd.read_csv(f"{DRIVE}/evals_summary.csv", dtype={"seq": str, "run": str, "checkpoint": str})
sp = e[(e.eval_type == "sparsity") & e.run.isin(RUNS.run)].copy()
sp["beams"] = sp["beams"].astype(int)
sp.to_csv(f"{DRIVE}/sparsity_all_runs.csv", index=False)
print("->", f"{DRIVE}/sparsity_all_runs.csv")
for mode in SP["modes"]:
    t = sp[sp["mode"] == mode].pivot_table(index=["method", "run", "checkpoint"], columns=["beams", "seq"],
                                           values="R@1").sort_index(axis=1, level=0, ascending=False)
    print({"both": "MODE both: map and queries sparse", "query": "MODE query: map 64 beams, queries sparse"}[mode])
    display(t.round(2))
print("Mean Recall@1 over the sequences")
display(sp.pivot_table(index=["method", "run"], columns=["mode", "beams"], values="R@1", aggfunc="mean")
          .sort_index(axis=1, level=[0, 1], ascending=[True, False]).round(2))
for r in RUNS.itertuples():
    png = glob.glob(f"{r.run_dir}/eval/sparsity/*/sparsity.png")
    if png:
        display(Image(sorted(png)[-1]))